In [ ]:
# Install
%pip install -e ..
# This kiiinda works?
%load_ext autoreload

In [ ]:
# Imports and Setup
%autoreload 2
import logging
import ezregex as ez
import re
from ezregex.invert import Inverter
import re._compiler
import re._parser as sre
import re._parser
from pprint import pformat
from rich.pretty import pprint
from collections.abc import Mapping

logging.basicConfig(level=logging.DEBUG)
original_handle = Inverter._handle


In [ ]:
# Cast function, before I realized SubPattern.dump() exists

# Just freaking cast all iterables (like sre.SubPattern) to lists
def cast(obj):
    try:
        iterable = iter(obj)
    except TypeError:
        if type(obj) == int:
            try:
                # return chr(obj), obj
                return obj
            except ValueError: pass
        return obj
    else:
        return [cast(i) for i in obj]

# pprint(live_log[0][0])
# pprint(cast(live_log[0][0]))

In [ ]:
def castSubPatterns(obj):
    try:
        _ = iter(obj)
    except TypeError:
        if type(obj) == int:
            try:
                # return chr(obj), obj
                return obj
            except ValueError: pass
        return obj
    else:
        return [cast(i) for i in obj]

In [ ]:
# Compile a partial AST

custom = sre.parse(r"\w*")
# ll = live_log[0]

# Mimmics re._compile.compile()
# Compile from an existing AST, instead of a string, so we can test if the partially deconstructed
# AST matches the partial AST it was deconstructed from
# FOR DEBUGGING ONLY
def compile_ast(pattern, flags=re.NOFLAG):
    if not isinstance(pattern, sre.SubPattern):
        return

    p = pattern
    code = re._compiler._code(p, re.NOFLAG)

    # map in either direction -- I don't know what this means or does
    groupindex = p.state.groupdict
    indexgroup = [None] * p.state.groups
    for k, i in groupindex.items():
        indexgroup[i] = k

    return re._compiler._sre.compile(
        # We don't have the actual string, but I think it's just for display, it seems to still work
        "", re.NOFLAG | p.state.flags, code,
        p.state.groups-1,
        groupindex, tuple(indexgroup)
        )

if (partial := compile_ast(custom)):
    partial.search('asdf')

In [ ]:
# Monkey patch _handle
# Don't rerun the imports cell after this, or it will enter an infinite loop!
live_log = []

def new_handle(self, pattern, amt=1, opposite=False):
    logging.debug(f'Handling {pattern} * {amt}')
    live_log.append((pattern, amt, opposite))
    rtn = original_handle(self, pattern, amt, opposite)
    logging.debug(f'Pattern is currently: "{rtn}"')
    # Check that the partial ast we just deconstructed matches the output we just got
    if (partial := compile_ast(pattern, re.NOFLAG)):
        if not (match := partial.search(rtn)):
            logging.warning('Pattern failed to match!')
        else:
            if match.span() != (0, len(rtn)):
                logging.warning('Pattern matched partially!')
            else:
                logging.debug('Valid!')
    return rtn

Inverter._handle = new_handle

In [ ]:
# Invert a partial AST directly to see if it's the issue

# Mimics Inverter.invert_re_parser()
inv = Inverter('')
inv.groups = {}
inv.seed=0.5569052451219174
def sub(l):
    return sre.SubPattern(sre.State(), data=l)
# You can copy these from the log, but be sure to add sre. to the tags, and cast the lists to SubPatterns
# partial_ast = [(sre.IN, [(sre.RANGE, (97, 122)), (sre.RANGE, (48, 57)), (sre.LITERAL, 45)])]
partial_ast = [(sre.MAX_REPEAT, (0, sre.MAXREPEAT, sub([(sre.IN, [(sre.RANGE, (97, 122)), (sre.RANGE, (48, 57)), (sre.LITERAL, 45)])]))), (sre.IN, [(sre.RANGE, (97, 122)), (sre.RANGE, (48, 57))])]

pattern = re._parser.SubPattern(re._parser.State(), data=partial_ast)
# Moneky patched version
for _ in range(1):
    inv._handle(pattern)


In [9]:
# Inverts a pattern directly, with a seed
import ezregex as ez

# pattern = ez.email
# pattern = r"(?:df){3}"
# pattern = r'(?:\w+\s*,\s*)?(\w+),?\s*'
pattern = r"(a??) a*? a{3,}? ab{4,7}?"

# Mimics Inverter.invert_re_parser()
inv = Inverter('')
inv.groups = {}
inv.seed=0.5569052451219174

# Moneky patched version
for _ in range(1):
    inv._handle(sre.parse(str(pattern)))
print(str(pattern))
print(sre.parse(str(pattern)))


DEBUG:root:Handling [(SUBPATTERN, (1, 0, 0, [(MIN_REPEAT, (0, 1, [(LITERAL, 97)]))])), (LITERAL, 32), (MIN_REPEAT, (0, MAXREPEAT, [(LITERAL, 97)])), (LITERAL, 32), (MIN_REPEAT, (3, MAXREPEAT, [(LITERAL, 97)])), (LITERAL, 32), (LITERAL, 97), (MIN_REPEAT, (4, 7, [(LITERAL, 98)]))] * 1
DEBUG:root:Handling [(MIN_REPEAT, (0, 1, [(LITERAL, 97)]))] * 1
DEBUG:root:Handling [(LITERAL, 97)] * 1
DEBUG:root:Literal: a * 1
DEBUG:root:Pattern is currently: "a"
DEBUG:root:Valid!
DEBUG:root:Pattern is currently: "a"
DEBUG:root:group: 1, num: 0, num2: 0 -> s: a -- sub: [(MIN_REPEAT, (0, 1, [(LITERAL, 97)]))]
DEBUG:root:Literal:   * 1
DEBUG:root:Handling [(LITERAL, 97)] * 1
DEBUG:root:Literal: a * 1
DEBUG:root:Pattern is currently: "a"
DEBUG:root:Valid!
DEBUG:root:Literal:   * 1
DEBUG:root:Handling [(LITERAL, 97)] * 1
DEBUG:root:Literal: a * 1
DEBUG:root:Pattern is currently: "a"
DEBUG:root:Valid!
DEBUG:root:Literal:   * 1
DEBUG:root:Literal: a * 1
DEBUG:root:Handling [(LITERAL, 98)] * 1
DEBUG:root:Lite

(a??) a*? a{3,}? ab{4,7}?
[(SUBPATTERN, (1, 0, 0, [(MIN_REPEAT, (0, 1, [(LITERAL, 97)]))])), (LITERAL, 32), (MIN_REPEAT, (0, MAXREPEAT, [(LITERAL, 97)])), (LITERAL, 32), (MIN_REPEAT, (3, MAXREPEAT, [(LITERAL, 97)])), (LITERAL, 32), (LITERAL, 97), (MIN_REPEAT, (4, 7, [(LITERAL, 98)]))]


In [10]:
# Show an AST for a particular regex string
sre.parse(r"(a??) a*? a{3,}? ab{4,7}?").dump()

SUBPATTERN 1 0 0
  MIN_REPEAT 0 1
    LITERAL 97
LITERAL 32
MIN_REPEAT 0 MAXREPEAT
  LITERAL 97
LITERAL 32
MIN_REPEAT 3 MAXREPEAT
  LITERAL 97
LITERAL 32
LITERAL 97
MIN_REPEAT 4 7
  LITERAL 98


In [16]:
# Just figure out what it's supposed to match
re.search(r"(a??) a*? a{3,}? ab{4,7}?", 'a aaaaa aaa abbbbbb')

<re.Match object; span=(0, 17), match='a aaaaa aaa abbbb'>

In [17]:
# Run some tests
live_log.clear()

print(ez.invert(ez.raw(r'(?:df){3}'), backend='re_parser'))

DEBUG:root:re_parser attempt #1 with seed 0.41139557224306644...
DEBUG:root:Literal: d * 1
DEBUG:root:Literal: f * 1
INFO:root:Found using re_parser


dfdfdf


In [20]:
# Disassemble a regex
# Not actually all that useful, but cool, for sure

# Dissassemble a regex
re._compiler.dis(re._compiler._code(sre.parse(r"[a-z0-9!#$%&'*+/=?^_`{|}~-]+"), re.NOFLAG))
# Dissassemble the live log
# re._compiler.dis(re._compiler._code(live_log[0][0], re.NOFLAG))

 0. INFO 4 0b0 1 MAXREPEAT (to 5)
 5: REPEAT_ONE 16 1 MAXREPEAT (to 22)
 9.   IN 11 (to 21)
11.     CHARSET [0x00000000, 0xa3ffacfa, 0xc0000000, 0x7fffffff, 0x00000000, 0x00000000, 0x00000000, 0x00000000]
20.     FAILURE
21:   SUCCESS
22: SUCCESS


In [ ]:
# Also kinda cool, not all that useful
def graphviz_nested(obj):
    dot = Digraph()

    seen_containers = {}
    next_id = 0

    def new_node():
        nonlocal next_id
        nid = f"n{next_id}"
        next_id += 1
        return nid

    def visit(obj):
        # Primitive values: always make a fresh node
        if isinstance(obj, (str, bytes, int, float, bool, type(None))):
            nid = new_node()
            dot.node(nid, repr(obj))
            return nid

        oid = id(obj)

        # Containers/objects: preserve identity
        if oid in seen_containers:
            return seen_containers[oid]

        nid = new_node()
        seen_containers[oid] = nid

        dot.node(nid, type(obj).__name__)

        if isinstance(obj, Mapping):
            for k, v in obj.items():
                k_id = visit(k)
                v_id = visit(v)
                dot.edge(nid, k_id, label="key")
                dot.edge(nid, v_id, label="value")

        elif isinstance(obj, (list, tuple)):
            for i, child in enumerate(obj):
                child_id = visit(child)
                dot.edge(nid, child_id, label=str(i))

        elif isinstance(obj, (set, frozenset)):
            for child in obj:
                child_id = visit(child)
                dot.edge(nid, child_id)

        elif hasattr(obj, "__dict__"):
            for name, value in vars(obj).items():
                child_id = visit(value)
                dot.edge(nid, child_id, label=name)

        return nid

    visit(obj)
    return dot

graphviz_nested(cast(live_log[0][0]))